# MoRE+ cookbook — knowledge injection on **every** backend

**MoRE+** (decoupled mixture-of-experts) turns each fact into its own tiny LoRA
expert on the model's *final-block FFN* (`down_proj`). A training-free **BM25
router** picks the best expert per question; its collapsed weight delta is merged
into that one FFN weight for the answer — which keeps the KV-cache valid (only a
post-attention weight changes) — then restored. The base is untouched at rest.

This notebook runs the **same** MoRE+ recipe on whichever backend you're on:

| backend | hardware | model picked |
|---|---|---|
| `mlx`   | Apple Silicon | `mlx-community/Qwen2.5-0.5B-Instruct-bf16` |
| `torch` | NVIDIA GPU (Colab) | `Qwen/Qwen2.5-0.5B-Instruct` |
| `torch` | CPU (any laptop) | `Qwen/Qwen2.5-0.5B-Instruct` |

> MoRE+ merges into a **real** weight, so it needs an **unquantized** base
> (bf16/fp16) — not a 4-bit repo.

In [ ]:
# Install shadowLM (batteries included: torch, transformers, trl, peft, ...).
# On Apple Silicon the mlx backend is pulled in automatically.
# Colab / fresh env:
#   !pip install shadowlm
# From a clone of this repo:
#   !pip install -e .

## 1. Pick the backend + model automatically

`slm.load` resolves the backend for your hardware. We pick an unquantized base to
match, and scale the per-expert step budget to the device (CPU gets fewer).

In [ ]:
import platform
import shadowlm as slm

APPLE = platform.system() == 'Darwin' and platform.machine() == 'arm64'
MODEL = ('mlx-community/Qwen2.5-0.5B-Instruct-bf16' if APPLE
         else 'Qwen/Qwen2.5-0.5B-Instruct')

# To force a backend (e.g. test the CPU path on a GPU box):
#   model = slm.load(MODEL, backend='torch', device='cpu')
model = slm.load(MODEL)
backend = model._backend.name
device = getattr(model._backend, 'device', '?')
STEPS = 60 if device == 'cpu' else 80   # steps per expert (precise numerals want more)
print(f'backend={backend} · device={device} · steps/expert={STEPS}')

## 2. The knowledge base

A handful of Lyzr support facts the base model has never seen. Self-contained, so
the notebook runs anywhere. One expert is trained per row.

In [ ]:
kb = slm.Dataset.from_list([
    {'messages': [{'role': 'user', 'content': 'Which Lyzr agent is the marketer?'},
                  {'role': 'assistant', 'content': "Skott is Lyzr's marketing agent."}]},
    {'messages': [{'role': 'user', 'content': 'Which Lyzr agent handles sales development?'},
                  {'role': 'assistant', 'content': "Jazon is Lyzr's SDR (sales development) agent."}]},
    {'messages': [{'role': 'user', 'content': 'Which Lyzr agent covers HR?'},
                  {'role': 'assistant', 'content': "Diane is Lyzr's HR agent."}]},
    {'messages': [{'role': 'user', 'content': 'Where is Lyzr headquartered?'},
                  {'role': 'assistant', 'content': 'Lyzr is headquartered in Jersey City, New Jersey.'}]},
])
print(len(kb.rows), 'facts ·', kb.format)

# Held-out phrasings (not the training questions) + the keyword each answer must recall.
probes = [
    ('Who is the marketing agent at Lyzr?',     'Skott'),
    ("Name Lyzr's sales development agent.",     'Jazon'),
    ('Which Lyzr agent does human resources?',   'Diane'),
    ('What city is Lyzr based in?',              'Jersey City'),
]

## 3. Before — the base model has no idea (it guesses or refuses)

In [ ]:
for q, _ in probes:
    a = model.generate(q, max_new_tokens=40, temperature=0.0).strip().replace(chr(10), ' ')
    print(f'Q: {q}\nA: {a}\n')

## 4. Train MoRE+

One final-FFN expert per fact, base frozen. `k=1` routes to the single best expert
(summing several into one FFN weight interferes); `lora_r == lora_alpha` keeps the
merge at scaling 1.0. Same call on every backend.

In [ ]:
run = model.finetune(
    kb,
    method='more_plus',
    more_plus_expert_steps=STEPS,
    more_plus_k=1,
    lora_r=8, lora_alpha=8,
)
print('final loss:', run.loss)

## 5. After — it recalls the injected facts

Each question is BM25-routed to its expert, whose delta is merged into the final
FFN just for that answer, then restored.

In [ ]:
hits = 0
for q, kw in probes:
    a = model.generate(q, max_new_tokens=40, temperature=0.0).strip().replace(chr(10), ' ')
    ok = kw.lower() in a.lower()
    hits += ok
    print(f"{'✓' if ok else '✗'} Q: {q}\n   A: {a}\n")
print(f'recalled {hits}/{len(probes)} facts')
assert hits >= len(probes) - 1, 'MoRE+ should recall almost every fact'

## 6. Cache-safe + stateless

The merge touches only a post-attention weight, and every call restores the base
from a snapshot — so generation is deterministic and drift-free. Same prompt,
twice, byte-identical:

In [ ]:
q = probes[0][0]
a1 = model.generate(q, max_new_tokens=24, temperature=0.0)
a2 = model.generate(q, max_new_tokens=24, temperature=0.0)
print('identical across calls (no drift):', a1 == a2)
assert a1 == a2

## 7. Save + reload — experts and router travel with the adapter

The adapter dir holds the expert deltas (one safetensors file) and the BM25 index.
Reload it on any backend; the base is fetched by name.

In [ ]:
path = run.checkpoint
import os; print('artifacts:', sorted(os.listdir(path)))

fresh = slm.load(MODEL, adapter=path)
q, kw = probes[1]
a = fresh.generate(q, max_new_tokens=40, temperature=0.0).strip().replace(chr(10), ' ')
print(f'Q: {q}\nA: {a}')
assert kw.lower() in a.lower()

## Notes

- **k**: 1 is the clean default (route to the best expert). Raise it only when facts
  genuinely combine — summing several deltas into one FFN weight can interfere.
- **steps/expert**: categorical facts (names, places) recall at 60–80; exact
  numerals (prices, dates) often want 120+.
- **group_size**: fold N rows into one expert when they describe the same thing.
- Same recipe, same checkpoint format on **mlx** and **torch (CUDA/CPU)** — train
  on a laptop, ship to a GPU, or vice-versa.